# Subqueries and CTEs

Some questions cannot be answered in a single, flat query. "Show me users
who listen more than average" -- what is average? You need to calculate it
first, then use that result in a filter. That means a query within a query.

Subqueries and Common Table Expressions (CTEs) are two ways to build these
layered queries. The recommendation engine at Wavelength uses both
extensively.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/wavelength.sqlite

## Scalar subqueries

A scalar subquery returns exactly one value -- one row, one column. You can
use it anywhere a single value is expected, most commonly in a WHERE clause.

Let's find the average number of listens per user, then find users who
listen more than that average. We could copy the number into our query,
but that is fragile -- the average changes as new data arrives. Instead,
we use a subquery.

In [ ]:
%%sql
SELECT user_id, COUNT(*) AS listen_count
FROM listens
GROUP BY user_id
HAVING COUNT(*) > (
    SELECT AVG(listen_count)
    FROM (
        SELECT user_id, COUNT(*) AS listen_count
        FROM listens
        GROUP BY user_id
    )
)
ORDER BY listen_count DESC
LIMIT 15;

The subquery in the HAVING clause calculates the average at query time.
Every time this runs, it uses the current data. No hard-coded numbers.

You can also use scalar subqueries in SELECT to add computed reference
values to every row.

In [ ]:
%%sql
SELECT
    title,
    play_count,
    (SELECT ROUND(AVG(play_count), 0) FROM tracks WHERE play_count IS NOT NULL) AS avg_play_count
FROM tracks
WHERE play_count IS NOT NULL
ORDER BY play_count DESC
LIMIT 10;

Every row now shows the overall average alongside its own play count,
making it easy to see which tracks are above or below average.

## Subqueries in FROM (derived tables)

A subquery in the FROM clause creates a temporary result set that you can
query just like a regular table. This is sometimes called a derived table.

In [ ]:
%%sql
SELECT
    artist_summary.artist_name,
    artist_summary.album_count,
    artist_summary.total_tracks
FROM (
    SELECT
        a.name AS artist_name,
        COUNT(DISTINCT al.album_id) AS album_count,
        COUNT(DISTINCT t.track_id) AS total_tracks
    FROM artists a
    LEFT JOIN albums al ON a.artist_id = al.artist_id
    LEFT JOIN tracks t ON a.artist_id = t.artist_id
    GROUP BY a.name
) AS artist_summary
WHERE artist_summary.album_count > 0
ORDER BY artist_summary.total_tracks DESC
LIMIT 10;

The inner query does the heavy lifting -- joining and aggregating. The outer
query filters and sorts. But notice how the nesting is already getting hard
to read. We will fix that with CTEs shortly.

## Correlated subqueries

A correlated subquery references a column from the outer query. It runs
once for every row of the outer query, which makes it powerful but
potentially slow on large datasets. Let's find each artist's most-played
track.

In [ ]:
%%sql
SELECT
    a.name AS artist,
    t.title AS top_track,
    t.play_count
FROM tracks t
INNER JOIN artists a ON t.artist_id = a.artist_id
WHERE t.play_count = (
    SELECT MAX(t2.play_count)
    FROM tracks t2
    WHERE t2.artist_id = t.artist_id
)
ORDER BY t.play_count DESC
LIMIT 15;

Look at the subquery: `WHERE t2.artist_id = t.artist_id`. The `t.artist_id`
comes from the outer query. For each track, the subquery finds the maximum
play count for that track's artist. If the track's count matches, it is the
top track.

This is elegant but can be slow. On our small dataset, that is fine. On
millions of rows, you would want a different approach (we will see window
functions in the next notebook).

## EXISTS: checking for related rows

EXISTS returns TRUE if a subquery returns at least one row. It is very
efficient because the database can stop searching as soon as it finds a
single match.

In [ ]:
%%sql
SELECT a.name, a.genre
FROM artists a
WHERE EXISTS (
    SELECT 1
    FROM tracks t
    INNER JOIN playlist_tracks pt ON t.track_id = pt.track_id
    WHERE t.artist_id = a.artist_id
)
ORDER BY a.name
LIMIT 15;

The `SELECT 1` inside EXISTS is conventional -- EXISTS only cares whether
any rows come back at all. You can also use `NOT EXISTS` to find the
opposite.

In [ ]:
%%sql
SELECT a.name, a.genre
FROM artists a
WHERE NOT EXISTS (
    SELECT 1
    FROM tracks t
    INNER JOIN playlist_tracks pt ON t.track_id = pt.track_id
    WHERE t.artist_id = a.artist_id
)
ORDER BY a.name;

## Common Table Expressions (CTEs)

Now for the part that will change how you write SQL.

Let's build a complex query: "For each genre, find the user who has
listened to the most tracks in that genre." Here it is as nested
subqueries.

In [ ]:
%%sql
SELECT
    genre_top.genre,
    u.username AS top_listener,
    genre_top.listen_count
FROM (
    SELECT
        genre_listens.genre,
        genre_listens.user_id,
        genre_listens.listen_count
    FROM (
        SELECT
            t.genre,
            l.user_id,
            COUNT(*) AS listen_count
        FROM listens l
        INNER JOIN tracks t ON l.track_id = t.track_id
        GROUP BY t.genre, l.user_id
    ) AS genre_listens
    WHERE genre_listens.listen_count = (
        SELECT MAX(gl2.listen_count)
        FROM (
            SELECT t2.genre, l2.user_id, COUNT(*) AS listen_count
            FROM listens l2
            INNER JOIN tracks t2 ON l2.track_id = t2.track_id
            GROUP BY t2.genre, l2.user_id
        ) AS gl2
        WHERE gl2.genre = genre_listens.genre
    )
) AS genre_top
INNER JOIN users u ON genre_top.user_id = u.user_id
ORDER BY genre_top.listen_count DESC;

That works. It also makes your eyes hurt. Subqueries nested three or four
levels deep are extremely hard to read, debug, and maintain. You have to
read from the inside out, holding intermediate results in your head.

This is exactly what CTEs solve. A CTE is defined with the `WITH` keyword.
It lets you name a subquery and reference it later, like creating a
temporary variable. Let's rewrite that monster.

In [ ]:
%%sql
WITH genre_listens AS (
    -- Step 1: Count listens per user per genre
    SELECT
        t.genre,
        l.user_id,
        COUNT(*) AS listen_count
    FROM listens l
    INNER JOIN tracks t ON l.track_id = t.track_id
    GROUP BY t.genre, l.user_id
),
genre_max AS (
    -- Step 2: Find the max listen count per genre
    SELECT genre, MAX(listen_count) AS max_listens
    FROM genre_listens
    GROUP BY genre
),
top_listeners AS (
    -- Step 3: Find the user(s) who hit that max
    SELECT gl.genre, gl.user_id, gl.listen_count
    FROM genre_listens gl
    INNER JOIN genre_max gm
        ON gl.genre = gm.genre
        AND gl.listen_count = gm.max_listens
)
-- Step 4: Add username and present results
SELECT
    tl.genre,
    u.username AS top_listener,
    tl.listen_count
FROM top_listeners tl
INNER JOIN users u ON tl.user_id = u.user_id
ORDER BY tl.listen_count DESC;

Same result. Dramatically more readable. Each CTE has a name and a clear
purpose. You can read it top to bottom like a recipe:

1. Count listens per user per genre
2. Find the maximum per genre
3. Match users to that maximum
4. Add usernames and display

This is the moment most engineers fall in love with CTEs. The logic is
identical to the nested version, but it reads like a story instead of a
puzzle.

## Chaining multiple CTEs

You can define as many CTEs as you need, separated by commas after the
initial `WITH`. Each CTE can reference the ones defined before it.

Let's build a user engagement summary: for each user, show their total
listens, playlist count, and whether they are above or below average.

In [ ]:
%%sql
WITH user_listens AS (
    SELECT user_id, COUNT(*) AS total_listens
    FROM listens
    GROUP BY user_id
),
user_playlists AS (
    SELECT user_id, COUNT(*) AS playlist_count
    FROM playlists
    GROUP BY user_id
),
avg_listens AS (
    SELECT ROUND(AVG(total_listens), 1) AS avg_listen_count
    FROM user_listens
)
SELECT
    u.username,
    u.subscription_type,
    COALESCE(ul.total_listens, 0) AS total_listens,
    COALESCE(up.playlist_count, 0) AS playlists,
    al.avg_listen_count,
    CASE
        WHEN COALESCE(ul.total_listens, 0) > al.avg_listen_count THEN 'above average'
        ELSE 'below average'
    END AS engagement
FROM users u
LEFT JOIN user_listens ul ON u.user_id = ul.user_id
LEFT JOIN user_playlists up ON u.user_id = up.user_id
CROSS JOIN avg_listens al
ORDER BY total_listens DESC
LIMIT 20;

The `CROSS JOIN avg_listens` adds that single-row reference value to every
row. The `CASE WHEN ... THEN ... ELSE ... END` is SQL's equivalent of
an if-else.

## When to use subqueries vs CTEs

Both achieve the same thing. **Use a subquery** when it is small and
self-contained -- a one-line scalar subquery in a WHERE clause is fine.
**Use a CTE** when the logic has multiple steps, when the same intermediate
result is used more than once, or when the nesting goes beyond two levels.

In practice, experienced engineers lean towards CTEs for anything
non-trivial. The readability benefit is enormous.

## A practical example: the recommendation engine

The recommendation team wants to find "similar users" -- pairs who have
listened to at least 5 of the same tracks.

In [ ]:
%%sql
WITH user_tracks AS (
    SELECT DISTINCT user_id, track_id
    FROM listens
),
shared_tracks AS (
    SELECT
        ut1.user_id AS user_a,
        ut2.user_id AS user_b,
        COUNT(*) AS shared_count
    FROM user_tracks ut1
    INNER JOIN user_tracks ut2
        ON ut1.track_id = ut2.track_id
        AND ut1.user_id < ut2.user_id
    GROUP BY ut1.user_id, ut2.user_id
    HAVING COUNT(*) >= 5
)
SELECT
    u1.username AS user_a,
    u2.username AS user_b,
    st.shared_count AS tracks_in_common
FROM shared_tracks st
INNER JOIN users u1 ON st.user_a = u1.user_id
INNER JOIN users u2 ON st.user_b = u2.user_id
ORDER BY st.shared_count DESC
LIMIT 15;

Three CTEs, each with a clear job. The final query just does the
presentation. This is the kind of query you would find in a real
recommendation system's codebase.

## Exercises

### Exercise 1

Use a scalar subquery to find all tracks with a play count above the
overall average. Show the title, genre, and play count. Order by play count
descending, limit to 15.

### Exercise 2

Use EXISTS to find all users who have created at least one playlist. Show
the username and subscription type.

### Exercise 3

Write a CTE that calculates total listening time per user. Then use that
CTE to find the top 10 users by listening time, showing their username
and total hours listened.

### Exercise 4

Write a query with two chained CTEs:
1. First CTE: count listens per genre per user
2. Second CTE: find the average listens per genre

Then find users who listen to any genre more than twice the average for
that genre. Show the username, genre, their listen count, and the genre
average.

### Exercise 5

Use NOT EXISTS to find tracks that have never been listened to. Show the
track title and artist name.

## Summary

- **Scalar subqueries** return a single value for use in WHERE, SELECT, or
  HAVING
- **Derived tables** (subqueries in FROM) create temporary result sets
- **Correlated subqueries** reference the outer query and run once per row
- **EXISTS / NOT EXISTS** efficiently check for the presence or absence of
  related rows
- **CTEs** (`WITH ... AS`) name intermediate results and make complex
  queries readable
- Multiple CTEs can be chained, each referencing the ones before it
- Favour CTEs over deep nesting -- your future self will thank you

In the final notebook, we tackle window functions: the tool that lets you
calculate across rows without collapsing them.